In [0]:
########################################
# Narrated dimension build for Lesson 08.
#
# Builds dim_product, dim_store, and dim_date with verbose narration so the
# learner can follow what is happening at each step without reading the
# underlying Python.
#
# Skips dim_customer (the learner builds that one hands-on in the demo
# notebook to practice the pattern).
########################################

try:
    _catalog = my_catalog  # set by Classroom-Setup-Common (reliable)
except NameError:
    _catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
_schema  = "vintage_audio"


def _line(char: str = "-", width: int = 72) -> None:
    print(char * width, flush=True)


def _section(label: str) -> None:
    print(flush=True)
    _line("=")
    print(f"  {label}", flush=True)
    _line("=")


def _say(text: str) -> None:
    print(text, flush=True)


def _table_exists(catalog: str, schema: str, table: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE `{catalog}`.`{schema}`.`{table}`")
        return True
    except Exception:
        return False


# ----- Prerequisite check -----------------------------------------------------
_missing = [t for t in ["customers", "products", "stores"]
            if not _table_exists(_catalog, _schema, t)]

if _missing:
    _line("=")
    _say(f"  WARNING: missing source tables: {', '.join(_missing)}")
    _say("  These are created by the M00 setup. Complete the 00 - Course")
    _say("  Overview notebook before continuing.")
    _line("=")
else:
    # ----- Begin narrated dimension build -------------------------------------
    _section("BUILDING THE REMAINING DIMENSIONS")
    _say("You already built dim_customer by hand. This step builds the other")
    _say("three dimension tables, which follow the exact same recipe:")
    _say("dim_product, dim_store, and dim_date.")
    _say("")
    _say("Each dimension follows the same recipe:")
    _say("  1. CREATE TABLE with a generated surrogate key.")
    _say("  2. Declare the surrogate key as the PRIMARY KEY.")
    _say("  3. INSERT business attributes from the source table.")

    # Step 1: dim_product
    _section("STEP 1 of 3: Build dim_product")
    _say("dim_product holds product attributes. The source table is products.")
    _say("We pick six columns: the natural key (product_id), three descriptive")
    _say("attributes (product_name, category, brand), and the list price.")
    _say("")
    _say("Creating dim_product with a generated surrogate key product_key ...")

    spark.sql(f"DROP TABLE IF EXISTS `{_catalog}`.`{_schema}`.dim_product")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_product (
            product_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            product_id   STRING,
            product_name STRING,
            category     STRING,
            brand        STRING,
            list_price   DOUBLE,
            CONSTRAINT pk_dim_product PRIMARY KEY (product_key)
        )
    """)

    _say("dim_product table created. It is empty.")
    _say("")
    _say("Inserting rows from the products table. Databricks generates")
    _say("product_key automatically for each new row, so the INSERT omits it.")

    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_product
            (product_id, product_name, category, brand, list_price)
        SELECT product_id, product_name, category, brand, list_price
        FROM `{_catalog}`.`{_schema}`.products
    """)

    n = spark.sql(
        f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.dim_product"
    ).collect()[0]["n"]
    _say(f"Inserted {n:,} rows into dim_product. Each row now has a unique")
    _say("product_key value generated by Databricks (1, 2, 3, ...).")

    # Step 2: dim_store
    _section("STEP 2 of 3: Build dim_store")
    _say("Same recipe applied to stores. We pick five columns: the natural key")
    _say("(store_id), the store name, and two location attributes (region and")
    _say("state).")
    _say("")
    _say("Creating dim_store with a generated surrogate key store_key ...")

    spark.sql(f"DROP TABLE IF EXISTS `{_catalog}`.`{_schema}`.dim_store")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_store (
            store_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            store_id   STRING,
            store_name STRING,
            region     STRING,
            state      STRING,
            CONSTRAINT pk_dim_store PRIMARY KEY (store_key)
        )
    """)

    _say("dim_store table created.")
    _say("")
    _say("Inserting rows from the stores table ...")

    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_store
            (store_id, store_name, region, state)
        SELECT store_id, store_name, region, state
        FROM `{_catalog}`.`{_schema}`.stores
    """)

    n = spark.sql(
        f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.dim_store"
    ).collect()[0]["n"]
    _say(f"Inserted {n:,} rows into dim_store.")

    # Step 3: dim_date
    _section("STEP 3 of 3: Build dim_date")
    _say("dim_date is the odd one out. There is no source table for it: dates")
    _say("are universal, not company-specific data. Every warehouse generates")
    _say("its own date dimension procedurally.")
    _say("")
    _say("The pattern: pick a date range generous enough to cover all current")
    _say("and future fact rows, then generate one row per day. For Vintage")
    _say("Audio we generate 2023 through 2026 (1,461 days).")
    _say("")
    _say("Creating dim_date with a generated surrogate key date_key ...")

    spark.sql(f"DROP TABLE IF EXISTS `{_catalog}`.`{_schema}`.dim_date")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_date (
            date_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            full_date DATE,
            year      INT,
            quarter   INT,
            CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
        )
    """)

    _say("dim_date table created.")
    _say("")
    _say("Generating one row per day from 2023-01-01 to 2026-12-31 ...")

    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_date
            (full_date, year, quarter)
        SELECT
            d                  AS full_date,
            YEAR(d)            AS year,
            QUARTER(d)         AS quarter
        FROM (
            SELECT DATE_ADD(DATE '2023-01-01', CAST(id AS INT)) AS d
            FROM range(0, 1461)
        )
    """)

    n = spark.sql(
        f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.dim_date"
    ).collect()[0]["n"]
    _say(f"Inserted {n:,} rows into dim_date.")

    # Finish
    _section("THREE MORE DIMENSIONS BUILT")
    _say("Schema status:")
    _say("")
    _say("  dim_customer  built  (you built this by hand)")
    _say("  dim_product   built  (with primary key)")
    _say("  dim_store     built  (with primary key)")
    _say("  dim_date      built  (with primary key)")
    _say("  fact_orders   NOT BUILT YET. Built next, after all dims exist.")
    _say("")
    _say("Continue scrolling to build the fact table.")